# Statistical significance checks

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import logging

from gsm_benchmarker.results_analyser import MultiVariantMultiModelResultsAnalyser
from gsm_benchmarker.results_analyser.prompt_effect_analyser import PromptEffectAnalyser
from gsm_benchmarker.results_analyser.plotting_utils import plot_models_odds_ratios, plot_bars_and_p_bars


logger = logging.getLogger('notebook')

plt.style.use('seaborn-v0_8-muted')
plt.style.use('seaborn-v0_8-darkgrid')

In [ ]:
pp = Path("../../../../data/gsm-symbolic/outputs").resolve()

p_standard = pp / "mini_20x50x4__14_11/final"
p_sep = pp / 'mini_sep_new__20x50__20_12/final'
p_code = pp / 'mini_code_output_20x50__05_12/final'

p_standard_full = pp / "default_full__06_02/final"
p_code_full = pp / 'code_full__09_02/final'

In [ ]:
mres_standard = MultiVariantMultiModelResultsAnalyser(p_standard)
# mres_sep = MultiVariantMultiModelResultsAnalyser(p_sep)
mres_code = MultiVariantMultiModelResultsAnalyser(p_code)

mres_standard_full = MultiVariantMultiModelResultsAnalyser(p_standard_full)
mres_code_full = MultiVariantMultiModelResultsAnalyser(p_code_full)


In [ ]:
# pea_sep = PromptEffectAnalyser(mres_standard, mres_sep, "Anti-babbling prompt")
pea_code = PromptEffectAnalyser(mres_standard, mres_code, "Code output prompt (pilot)")
pea_code_full = PromptEffectAnalyser(mres_standard_full, mres_code_full, "Code output prompt")

### Modelling question difficulty

In [ ]:
# _ = pea_code._baseline_mres.plot_question_difficulty_per_model()

In [ ]:
# _ = pea_code._baseline_mres.plot_question_difficulty_histogram()

## Question 1
Are the accuracy drops reported in the GSM-Symbolic paper actually significant?

Evaluating significance of accuracy drop on 'main' variant vs 'GSM8K' variant with GSM-Symbolic prompt.

### 1A: Pilot run
Checking on a subset of the benchmark first. Projecting results to the full dataset with alpha = 20%.

50/100 questions, 20/50 template variations -> 1000/5000 total questions in the 'main variant' (20% of the benchmark) (+ 50 questions in the original GSM8K variant)

In [ ]:
PROJECTED_ALPHA = 0.2

In [ ]:
mres_code_full._summary_data.index

In [ ]:
variant_effect_df = mres_standard.analyse_variant_effect('main')
variant_effect_df

In [ ]:
_, model_order_standard = plot_models_odds_ratios(variant_effect_df, 'standard', projected_alpha=PROJECTED_ALPHA, log_scale=True, sort_models=True)


In [ ]:
_, model_order_discounted = plot_models_odds_ratios(variant_effect_df, 'discounted', projected_alpha=PROJECTED_ALPHA, log_scale=True, sort_models=True)

In [ ]:
_ = plot_bars_and_p_bars(variant_effect_df, 'accuracy_drop', 'p_value', alpha=0.05, projected_alpha=0.2, model_order=model_order_standard)


#### Candidate models
Models which, based on the pilot results, are worth checking on the full dataset.

In [ ]:
candidate_models = variant_effect_df[variant_effect_df.p_value < PROJECTED_ALPHA].index.get_level_values('model').unique().tolist()
candidate_models

### 1B: Full benchmark
Re-running the tests on the full benchmark (100 + 5000 questions) with the identified candidate models.

In [ ]:
variant_effect_df_full = mres_standard_full.analyse_variant_effect('main', models=candidate_models)
variant_effect_df_full

In [ ]:
_, model_order_standard_full = plot_models_odds_ratios(
    variant_effect_df_full, 'standard', log_scale=True, sort_models=True)  # model_order=model_order_standard
_, model_order_discounted_full = plot_models_odds_ratios(
    variant_effect_df_full, 'discounted', log_scale=True, sort_models=True)  # model_order=model_order_discounted,

In [ ]:
_ = plot_bars_and_p_bars(variant_effect_df_full, 'accuracy_drop', 'p_value', alpha=0.05, model_order=model_order_standard_full)

## Question 2
Does code prompt remove the variant dependency?

Evaluating significance of accuracy drop on 'main' variant vs 'GSM8K' variant with code prompt.

In [ ]:
variant_effect_df_full_code = mres_code_full.analyse_variant_effect('main', models=candidate_models)
_, model_order_code_standard_full = plot_models_odds_ratios(
    variant_effect_df_full_code, 'standard', log_scale=True, sort_models=True)  # model_order=model_order_standard_full,
_, model_order_code_discounted_full = plot_models_odds_ratios(
    variant_effect_df_full_code, 'discounted', log_scale=True, sort_models=True)  # model_order=model_order_discounted_full,


In [ ]:
_ = plot_bars_and_p_bars(variant_effect_df_full_code, 'accuracy_drop', 'p_value', alpha=0.05, model_order=model_order_code_standard_full)

### Code prompt evaluation
Does code prompt result in significant performance changes?

Evaluating performance on 'main' variant with the code prompt vs GSM-Symbolic prompt.

In [ ]:
acc_change_df = pea_code_full.analyse_accuracy_change_significance(variant='main', models=candidate_models)
acc_change_df

In [ ]:
_, model_order_code_accuracy_change_standard = plot_models_odds_ratios(
    acc_change_df, 'standard', log_scale=True, sort_models=True)
_ = plot_models_odds_ratios(acc_change_df, 'discounted', log_scale=True, sort_models=True)

In [ ]:
_ = plot_bars_and_p_bars(
    acc_change_df, 'median_diff', 'p_value', colours=['lightgreen', 'seagreen'],
    model_order=model_order_code_accuracy_change_standard, ylabel0='Accuracy change (median)')